In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
import asyncio
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
openai_model_client = OpenAIChatCompletionClient(model='gpt-4o',api_key=api_key)



In [2]:
from typing import Text
from autogen_agentchat.conditions import MaxMessageTermination

agent_1  =AssistantAgent(name = "Writer_1",
                         model_client=openai_model_client,
                         system_message="You are a helpful assistant. Give output in less than 30 words.")

agent_2  =AssistantAgent(name = "Writer_2",
                         model_client=openai_model_client,
                         system_message="You are a helpful assistant. Give output in less than 30 words.")

terminationCondition = MaxMessageTermination(max_messages=3)

agent_team = RoundRobinGroupChat(participants=[agent_1,agent_2],termination_condition=terminationCondition)

In [5]:
from autogen_agentchat.ui import Console
stream = agent_team.run_stream(task = "Write a poem about sea in 3 lines")
await Console(stream)

---------- TextMessage (user) ----------
Write a poem about sea in 3 lines
---------- TextMessage (Writer_1) ----------
Waves whisper secrets of azure dreams,   
Tides embrace golden sands in gentle dance,   
Eternal horizon kisses the sky's seams.
---------- TextMessage (Writer_2) ----------
Azure waves embrace golden shores,   
Eternal dance beneath wide skies,   
Secrets drift where seabirds soar.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 16, 2, 773616, tzinfo=datetime.timezone.utc), content='Write a poem about sea in 3 lines', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=36, completion_tokens=28), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 16, 5, 3658, tzinfo=datetime.timezone.utc), content="Waves whisper secrets of azure dreams,   \nTides embrace golden sands in gentle dance,   \nEternal horizon kisses the sky's seams.", type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=72, completion_tokens=22), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 16, 7, 129944, tzinfo=datetime.timezone.utc), content='Azure waves embrace golden shores,   \nEternal dance beneath wide skies,   \nSecrets drift where seabirds soar.', type='TextMessage')], stop_reason='Maximum number of messages 3 reached,

In [6]:
team_state = await agent_team.save_state()
team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a poem about sea in 3 lines',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "Waves whisper secrets of azure dreams,   \nTides embrace golden sands in gentle dance,   \nEternal horizon kisses the sky's seams.",
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'}]}},
   'message_buffer': [{'source': 'Writer_2',
     'models_usage': {'prompt_tokens': 72, 'completion_tokens': 22},
     'metadata': {},
     'created_at': datetime.datetime(2025, 6, 18, 20, 16, 7, 129944, tzinfo=datetime.timezone.utc),
     'content': 'Azure waves embrace golden shores,   \nEternal dance beneath wide skies,   \nSecrets drift where seabirds soar.',
     'type': 'TextMessage'}]},
  'Writer_2': {

In [9]:
await agent_team.reset()

In [10]:
stream = agent_team.run_stream(task = "What was the last poem you gave to me")
await Console(stream)

---------- TextMessage (user) ----------
What was the last poem you gave to me
---------- TextMessage (Writer_1) ----------
I'm sorry, but I can't recall or access previous interactions, including any poems I might have provided.
---------- TextMessage (Writer_2) ----------
I can't retrieve past interactions, so I'm unable to recall or review poems previously given to you.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 19, 19, 192631, tzinfo=datetime.timezone.utc), content='What was the last poem you gave to me', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=36, completion_tokens=20), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 19, 21, 99739, tzinfo=datetime.timezone.utc), content="I'm sorry, but I can't recall or access previous interactions, including any poems I might have provided.", type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=19), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 19, 22, 769003, tzinfo=datetime.timezone.utc), content="I can't retrieve past interactions, so I'm unable to recall or review poems previously given to you.", type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [11]:
await agent_team.load_state(team_state)
stream = agent_team.run_stream(task = "What was the last poem you gave to me")
await Console(stream)

---------- TextMessage (user) ----------
What was the last poem you gave to me
---------- TextMessage (Writer_1) ----------
Waves whisper secrets of azure dreams,   
Tides embrace golden sands in gentle dance,   
Eternal horizon kisses the sky's seams.
---------- TextMessage (Writer_2) ----------
Azure waves embrace golden shores,  
Eternal dance beneath wide skies,  
Secrets drift where seabirds soar.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 20, 8, 917080, tzinfo=datetime.timezone.utc), content='What was the last poem you gave to me', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=112, completion_tokens=28), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 20, 10, 156865, tzinfo=datetime.timezone.utc), content="Waves whisper secrets of azure dreams,   \nTides embrace golden sands in gentle dance,   \nEternal horizon kisses the sky's seams.", type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=148, completion_tokens=22), metadata={}, created_at=datetime.datetime(2025, 6, 18, 20, 20, 11, 559899, tzinfo=datetime.timezone.utc), content='Azure waves embrace golden shores,  \nEternal dance beneath wide skies,  \nSecrets drift where seabirds soar.', type='TextMessage')], stop_reason='Maximum number of messages 3 

In [1]:
import json

## save state to disk

with open("coding/team_state.json", "w") as f:
    json.dump(team_state, f)

## load state from disk
with open("coding/team_state.json", "r") as f:
    team_state = json.load(f)


FileNotFoundError: [Errno 2] No such file or directory: 'coding/team_state.json'